In [8]:

import pandas as pd
from pathlib import Path

DATA_PATH = "synthetic_covid_dataset.csv"
df = pd.read_csv(DATA_PATH)

clinical_features = [
    "age", "sex", "fever", "cough", "dyspnea",
    "fatigue", "headache", "hypertension", "diabetes",
    "cardiovascular_disease", "copd", "chronic_liver_disease",
    "chronic_kidney_disease", "cancer"
]

lab_features = [
    "wbc", "neutrophil", "lymphocyte", "hemoglobin",
    "platelet", "d_dimer", "crp", "albumin", "alt", "ast",
    "bilirubin", "potassium", "sodium", "creatinine",
    "ck", "ldh", "hbdh"
]

radiomics_features = [
    col for col in df.columns
    if col.startswith((
        "firstorder_", "glcm_", "glszm_",
        "glrlm_", "ngtdm_", "wavelet_"
    ))
]

target_features = ["ICU", "mechanical_ventilation", "death"]

feature_groups = {
    "Radiom": radiomics_features,
    "RadioClin": radiomics_features + clinical_features,
    "RadioClinLab": radiomics_features + clinical_features + lab_features
}

for name, features in feature_groups.items():
    print(f"{name}: {len(features)} features")

print("\nDataset shape:", df.shape)


Radiom: 120 features
RadioClin: 134 features
RadioClinLab: 151 features

Dataset shape: (2400, 155)


In [9]:

from sklearn.model_selection import train_test_split

X_all = df.copy()
Y = df[target_features].copy()

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.30,
    random_state=42,
    stratify=df["ICU"]
)

print("Training patients:", len(train_idx))
print("Testing patients:", len(test_idx))


Training patients: 1680
Testing patients: 720


In [12]:

import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from scipy import sparse

# Mount Google Drive if it is not already mounted
from google.colab import drive
drive.mount("/content/drive")

# Correct Google Drive output directory
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/data/processed"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Saving processed files to:", OUTPUT_DIR)

processed_groups = {}
preprocessors = {}

for group_name, features in feature_groups.items():

    X_group = df[features].copy()

    # Identify categorical columns using training data only
    categorical_cols = [
        col for col in features
        if (
            not pd.api.types.is_numeric_dtype(X_group.loc[train_idx, col])
            or X_group.loc[train_idx, col].nunique(dropna=True) <= 2
        )
    ]

    numeric_cols = [
        col for col in features
        if col not in categorical_cols
    ]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    transformers = []

    if numeric_cols:
        transformers.append(
            ("numeric", numeric_pipeline, numeric_cols)
        )

    if categorical_cols:
        transformers.append(
            ("categorical", categorical_pipeline, categorical_cols)
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    X_train = X_group.loc[train_idx]
    X_test = X_group.loc[test_idx]

    # Fit only on training data
    X_train_processed = preprocessor.fit_transform(X_train)

    # Apply the same transformation to test data
    X_test_processed = preprocessor.transform(X_test)

    processed_groups[group_name] = {
        "X_train": X_train_processed,
        "X_test": X_test_processed,
        "features": features
    }

    preprocessors[group_name] = preprocessor

    # Save dense or sparse matrices
    def save_matrix(matrix, path):
        if sparse.issparse(matrix):
            sparse.save_npz(str(path) + ".npz", matrix)
        else:
            np.save(str(path) + ".npy", matrix)

    save_matrix(
        X_train_processed,
        OUTPUT_DIR / f"{group_name}_X_train"
    )

    save_matrix(
        X_test_processed,
        OUTPUT_DIR / f"{group_name}_X_test"
    )

    joblib.dump(
        preprocessor,
        OUTPUT_DIR / f"{group_name}_preprocessor.joblib"
    )

    print(
        f"{group_name}: "
        f"train={X_train_processed.shape}, "
        f"test={X_test_processed.shape}"
    )

# Save target splits to the same directory
Y.loc[train_idx].to_csv(
    OUTPUT_DIR / "Y_train.csv",
    index=False
)

Y.loc[test_idx].to_csv(
    OUTPUT_DIR / "Y_test.csv",
    index=False
)

print("\nAll feature groups processed and saved.")
print("\nSaved files:")
for file in sorted(OUTPUT_DIR.iterdir()):
    print(file.name)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving processed files to: /content/drive/MyDrive/Colab Notebooks/data/processed
Radiom: train=(1680, 120), test=(720, 120)
RadioClin: train=(1680, 147), test=(720, 147)
RadioClinLab: train=(1680, 164), test=(720, 164)

All feature groups processed and saved.

Saved files:
RadioClinLab_X_test.npy
RadioClinLab_X_train.npy
RadioClinLab_preprocessor.joblib
RadioClin_X_test.npy
RadioClin_X_train.npy
RadioClin_preprocessor.joblib
Radiom_X_test.npy
Radiom_X_train.npy
Radiom_preprocessor.joblib
Y_test.csv
Y_train.csv


In [11]:

for group_name, data in processed_groups.items():
    X_train = data["X_train"]
    X_test = data["X_test"]

    print(f"\n{group_name}")
    print("Training shape:", X_train.shape)
    print("Testing shape:", X_test.shape)
    print("Original feature count:", len(data["features"]))



Radiom
Training shape: (1680, 120)
Testing shape: (720, 120)
Original feature count: 120

RadioClin
Training shape: (1680, 147)
Testing shape: (720, 147)
Original feature count: 134

RadioClinLab
Training shape: (1680, 164)
Testing shape: (720, 164)
Original feature count: 151
